# v2.08 — v2.07 with the B2 "true data" label (2 classes, next day; validation only)

**One change on top of v2.07** (Siwat, 2026-11-03, `configs/v2_08.json`; label decision `claude/decision-2026-11-03-binary-label-b2.md`):
- **What changes:** the label. **0 = Down** if tomorrow's raw close is below today's, **1 = Not-down** (up or unchanged) otherwise, inside one price segment. Raw Siamchart closes (no split / dividend adjustment). The trend head has 2 outputs.
- **What stays:** everything else is v2.07: 1-day horizon only, no price head, news-robust training (p 0.2), plain cross-entropy (no class weights), kept epoch on the hold-out macro-F1, the same rows and seeds.

**Class balance:** Not-down is 56.5% of training rows and 56.6% of validation rows, so accuracy alone overstates skill (always "Not-down" scores about 0.566). Macro-F1 is the headline.

**Status:** a secondary analysis. The pre-registered primary label stays the 3-class k × σ label (success-band amendment rule 5).

**Runtime:** A100 or L4, Colab only (Linux). Runtime → Run all. About 1 h of training, then about 2 min for the comparison. No post-hoc step (Siwat: raw only).

**Adopt rule** (pre-registered, `claude/log-entry-2026-11-03-v2-08-design.md`), raw model, validation macro-F1 over the 2 classes, paired 95% date-block intervals:
- **(a)** v2.08 − XGBoost (B2 label, fit rows) entirely above 0;
- **(b)** v2.08 − random calls at the training imbalance (P(Not-down) = 0.565; 100 draws) entirely above 0.

Outputs: `Senior/results/v2xx/<GPU>/v2_08*`, `Senior/results/v2_08_vs_b2_baselines.json`. Test block sealed.

In [ ]:
!nvidia-smi -L
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {name}, {mem:.0f} GB')
GPU_TAG = 'A100' if 'A100' in name else 'L4' if 'L4' in name else None
assert GPU_TAG, 'stop: switch the runtime to an A100 or L4 GPU (Runtime -> Change runtime type)'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess, sys
from pathlib import Path
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, RES = f'{SENIOR}/prices', f'{SENIOR}/results'
OUT = f'{RES}/v2xx/{GPU_TAG}'
V208 = f'{OUT}/v2_08.json'
CMP = f'{RES}/v2_08_vs_b2_baselines.json'
for f in [f'{PRICE_DIR}/set50_union_2019_2025.csv', f'{PRICE_DIR}/yahoo_events_2019_2025.csv']:
    assert os.path.exists(f), f'stop: missing {f}'
os.makedirs(OUT, exist_ok=True)

def stream(args):
    # stream a script's output into the notebook and stop on failure
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if 'defined on' not in line and 'not writable' not in line and 'from_numpy' not in line:
            print(line, end='', flush=True)
    if p.wait() != 0:
        raise SystemExit(f'stop: {" ".join(args[1:3])} failed (exit {p.returncode}); see the lines above')

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
for f in ['configs/v2_08.json', 'scripts/train_v2xx.py', 'src/labels.py', 'src/metrics.py', 'src/posthoc.py', 'src/train_utils.py', 'src/run_config.py', 'tests/test_no_price_head.py', 'tests/test_one_horizon.py', 'tests/test_b2_label.py', 'src/proposal_model.py',
          'scripts/compare_b2.py', 'scripts/plot_v2xx_training.py']:
    assert os.path.exists(f), f'stop: push the v2.08 code first ({f} missing)'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the frozen copy'
sys.path.insert(0, 'src'); sys.path.insert(0, 'scripts')
import run_config as RC
cfg = RC.load('configs/v2_08.json')
print('v2.08 label:', cfg['protocol']['label_rule'], '| horizons:', cfg['protocol']['horizons'], '| price head:', cfg['model']['price_head'], '| loss:', json.dumps(cfg['loss']), '| training:', json.dumps(cfg['training']))

In [ ]:
# Tests (about 15 minutes); the notebook stops if any fails
for t in ['tests/test_v2xx.py', 'tests/test_v2xx_training.py', 'tests/test_news_blank.py', 'tests/test_grad_diag.py', 'tests/test_no_price_head.py', 'tests/test_one_horizon.py', 'tests/test_b2_label.py',
          'tests/test_step0_v2_03beta.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l and '    seed' not in l)[-2500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# 1. Train v2.08 (B2 label, 1-day horizon only, no price head): 3 seeds one after another
def done(path, n=3):
    try:
        r = json.load(open(path)); return len(r['runs']) == n and 'summary' in r
    except Exception:
        return False
if done(V208):
    print('v2.08: already trained')
else:
    stream(['python', '-u', 'scripts/train_v2xx.py', '--config', 'configs/v2_08.json', '--out', V208])

In [ ]:
# 2. Summary beside the B2 floors (computed in the run), and the learning curves
r = json.load(open(V208)); s, fl = r['summary']['model']['h1'], r['floors']['h1']
print('label', r['label']['rule'], r['label']['classes'], '| not-down share train', r['label']['train_class_shares'][1],
      '| fit / hold-out / val', [x[1] for x in r['label']['class_shares'].values()])
print(f"v2.08 macro-F1 {s['macro_f1']['mean']:.4f} ± {s['macro_f1']['std']:.4f} | accuracy {s['accuracy']['mean']:.4f} ± {s['accuracy']['std']:.4f}")
for k in ('majority_class', 'stratified_random_expected', 'trend_continues', 'trend_reverses'):
    print(f"  floor {k:28s} macro-F1 {fl[k]['macro_f1']:.4f}   accuracy {fl[k].get('accuracy', float('nan')):.4f}")
for x in r['runs']:
    print('seed', x['seed'], 'epochs', x['epochs_run'], 'kept', x['best_epoch'], 'hold-out F1', x['best_inner_es_macro_f1'],
          'val F1', x['val']['h1']['macro_f1'], 'confusion [[down],[not down]]', x['val']['h1']['confusion'],
          'weights', x['params']['total'], 'minutes', round(x['seconds'] / 60, 1))
if 'seed_ensemble' in r:
    print('seed ensemble', r['seed_ensemble']['val']['h1'])
from IPython.display import Image, display
import importlib, plot_v2xx_training as PT
for png in (PT.plot(Path(V208)), PT.plot_calls(Path(V208))):
    if png:
        print('saved', png); display(Image(filename=str(png)))

In [ ]:
# 3. v2.08 vs XGBoost (B2) and random at the B2 imbalance on the same rows, and the pre-registered adopt rule
stream(['python', 'scripts/compare_b2.py', '--run', V208, '--out', CMP])

**Reading the result**
- **ADOPT RULE** (last block): v2.08 must beat both XGBoost (B2) and random calls at the training imbalance, each with the paired interval entirely above 0. Raw model.
- Random at the imbalance has an expected macro-F1 of **0.5**; always "Not-down" scores about 0.37 macro-F1 but 0.566 accuracy.
- Not comparable with v2.01–v2.07 (different label).
- Send the printout to Claude.